In [50]:
import sympy as sp

In [51]:
# define independent variables 
x,y = sp.symbols('x, y', real=True)
xv = sp.Matrix([x,y])

# define the rhs of the governing equation
w = sp.symbols("omega")
u1,u2 = sp.symbols("u_x, u_y", real=True)
u1 = sp.Function("u_x")(x, y)
u2 = sp.Function("u_y")(x, y)
u = sp.Matrix([u1,u2])
grad_w = sp.Matrix([sp.Derivative(w,x), sp.Derivative(w,y)])
FU = -u.dot(grad_w)
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [52]:
# define q(t)
N = 2 #number of vortexes

q = sp.Matrix()

A = sp.Matrix()
L = sp.Matrix()
xc= sp.Matrix()
yc= sp.Matrix()
# r = sp.Matrix()

for i in range(N):
    A = sp.Matrix([A, sp.symbols("A_"+str(i+1), real=True)])
    L = sp.Matrix([L, sp.symbols("L_"+str(i+1), real=True, positive=True)])
    xc = sp.Matrix([xc, sp.symbols("x_c_"+str(i+1), real=True)])
    yc = sp.Matrix([yc, sp.symbols("y_c_"+str(i+1), real=True)])
    # r = sp.Matrix([r, sp.symbols("r_"+str(i+1), real=True, positive=True)])
    # r = sp.Matrix([r, sp.Function("r_"+str(i+1))(x, y, xc[i], yc[i])])

q = sp.Matrix([A, L, xc, yc])
# qr = sp.Matrix([A, L, r])

q

Matrix([
[  A_1],
[  A_2],
[  L_1],
[  L_2],
[x_c_1],
[x_c_2],
[y_c_1],
[y_c_2]])

In [53]:
# define the ansatz u_hat(x; q)
ansatz_gamma = 0
for i in range(N):
    ansatz_gamma = ansatz_gamma + A[i]*sp.exp(-(((x-xc[i])**2+(y-yc[i])**2))/L[i]**2)

ansatz_gamma

A_1*exp((-(x - x_c_1)**2 - (y - y_c_1)**2)/L_1**2) + A_2*exp((-(x - x_c_2)**2 - (y - y_c_2)**2)/L_2**2)

In [54]:
ansatz_u = sp.Matrix([
    sp.Derivative(ansatz_gamma,y).doit().simplify(),
    -sp.Derivative(ansatz_gamma, x).doit().simplify()
])

ansatz_u

Matrix([
[-(2*A_1*L_2**2*(y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L_1**2) + 2*A_2*L_1**2*(y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L_2**2))/(L_1**2*L_2**2)],
[ (2*A_1*L_2**2*(x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L_1**2) + 2*A_2*L_1**2*(x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L_2**2))/(L_1**2*L_2**2)]])

In [55]:
ansatz = (- sp.Derivative(ansatz_gamma, x, 2) - sp.Derivative(ansatz_gamma, y, 2)).doit()
ansatz.simplify()

4*(A_1*L_1**2*L_2**4*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L_1**2) - A_1*L_2**4*(x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L_1**2) - A_1*L_2**4*(y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L_1**2) + A_2*L_1**4*L_2**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L_2**2) - A_2*L_1**4*(x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L_2**2) - A_2*L_1**4*(y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L_2**2))/(L_1**4*L_2**4)

In [56]:
# compute partial derivatives du/dqi
dwdq = ansatz.diff(q)

dwdq.simplify()

Matrix([
[                                                       4*(L_1**2 - (x - x_c_1)**2 - (y - y_c_1)**2)*exp((-(x - x_c_1)**2 - (y - y_c_1)**2)/L_1**2)/L_1**4],
[                                                       4*(L_2**2 - (x - x_c_2)**2 - (y - y_c_2)**2)*exp((-(x - x_c_2)**2 - (y - y_c_2)**2)/L_2**2)/L_2**4],
[8*A_1*(-L_1**4 + 3*L_1**2*((x - x_c_1)**2 + (y - y_c_1)**2) - ((x - x_c_1)**2 + (y - y_c_1)**2)**2)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L_1**2)/L_1**7],
[8*A_2*(-L_2**4 + 3*L_2**2*((x - x_c_2)**2 + (y - y_c_2)**2) - ((x - x_c_2)**2 + (y - y_c_2)**2)**2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L_2**2)/L_2**7],
[                                     8*A_1*(x - x_c_1)*(2*L_1**2 - (x - x_c_1)**2 - (y - y_c_1)**2)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L_1**2)/L_1**6],
[                                     8*A_2*(x - x_c_2)*(2*L_2**2 - (x - x_c_2)**2 - (y - y_c_2)**2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L_2**2)/L_2**6],
[                                     8*A_1*(y - 

In [57]:
xmin,xmax = sp.symbols("x_{min}, x_{max}")
# define the inner product according to the problem
def inner_prod_H(f, g):
    ix = sp.integrate((f*g).expand(),(x, -sp.oo, sp.oo))
    return sp.integrate(ix.expand(),(y, -sp.oo, sp.oo)).expand()

In [60]:
inner_prod_H(dwdq[3], dwdq[3]).simplify()
# m00 = (dwdq[0]**2).simplify()

# m00i = sp.integrate(m00.expand(), (x, -sp.oo, +sp.oo))

16*pi*A_2**2/L_2**4

In [59]:
# construct the matrix M_ij = <du/dqi, du/dqj>_H
n = len(q)
M = sp.zeros(n, n)

for i in range(n):
    M[i, i] = inner_prod_H(dwdq[i], dwdq[i]).simplify()
    print(M[i, i])
    for j in range(i+1, n):
        M[i, j] = inner_prod_H(dwdq[i], dwdq[j]).simplify()
        M[j,i] = M[i,j]
        print(M[i, j])


4*pi/L_1**2


KeyboardInterrupt: 

In [ ]:
M

In [ ]:
FU

In [ ]:
# compute rhs from the ansatz
Fua = FU.subs(u1, ansatz_u[0]).subs(u2, ansatz_u[1]).subs(w, ansatz).doit()
Fua.simplify()

In [ ]:
# compute f
n = len(q)
f = sp.zeros(n, 1)

for i in range(n):
    f[i] = inner_prod_H(dwdq[i], Fua).simplify()
    print(f[i])

f

In [27]:
q_dot = M.inv()*f

q_dot.simplify()

In [ ]:
q_dot